# spaCy Transformers

We will learn spaCy Transformers, step by step.

Think of a Transformer like a super-smart reading robot.
- It reads your sentence
- Breaks words into tiny pieces (wordpieces)
- Turns each piece into numbers (vectors)
- Uses those numbers to understand meaning

In [63]:
import spacy
import numpy as np
from importlib.metadata import version

print("spacy:", spacy.__version__)
print("spacy-transformers:", version("spacy-transformers"))

spacy: 3.8.11
spacy-transformers: 1.3.9


## 1) Load a ready Transformer pipeline

We use a prebuilt English pipeline with transformer + NER.

In [64]:
nlp = spacy.load("en_core_web_trf")
text = "Apple is opening a new office in Riyadh for $1 billion."
doc = nlp(text)

print("Text:", doc.text)
print("Tokens:", [t.text for t in doc])

Text: Apple is opening a new office in Riyadh for $1 billion.
Tokens: ['Apple', 'is', 'opening', 'a', 'new', 'office', 'in', 'Riyadh', 'for', '$', '1', 'billion', '.']


## 2) Named Entities (real-world understanding)

The model can find names like companies, places, money, dates.

In [65]:
entities = [
    (ent.text, ent.label_, spacy.explain(ent.label_)) for ent in doc.ents
]

entities

[('Apple', 'ORG', 'Companies, agencies, institutions, etc.'),
 ('Riyadh', 'GPE', 'Countries, cities, states'),
 ('$1 billion', 'MONEY', 'Monetary values, including unit')]

## 3) Where transformer vectors live

Important:
- doc.tensor can be empty in transformer pipelines (only available in older version before 3.7)
- In newer version Use doc._.trf_data in spaCy transformers
- In newer version Hidden states are in last_hidden_layer_state.data

In [66]:
trf = doc._.trf_data
print("Available trf_data attrs:")
print([a for a in dir(trf) if not a.startswith("_")])

wp = trf.last_hidden_layer_state
print("Type:", type(wp))
print("Wordpiece data shape:", wp.data.shape)  # (num_wordpieces, hidden_size)

Available trf_data attrs:
['all_hidden_layer_states', 'all_outputs', 'embedding_layer', 'from_dict', 'last_hidden_layer_state', 'last_layer_only', 'num_outputs', 'to_dict']
Type: <class 'thinc.types.Ragged'>
Wordpiece data shape: (13, 768)


## 4) Build a sentence vector

We average all wordpiece vectors to make one sentence vector.

In [67]:
sentence_vec = wp.data.mean(axis=0)
print("Sentence vector shape:", sentence_vec.shape)

Sentence vector shape: (768,)


## 5) Compare sentence meaning with cosine similarity

Bigger score = more similar meaning.

In [68]:
def sent_vec(nlp, text):
    d = nlp(text)
    return d._.trf_data.last_hidden_layer_state.data.mean(axis=0)

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


In [69]:
s1 = "I love this movie."
s2 = "This film is fantastic."
s3 = "The weather is very hot today."

v1 = sent_vec(nlp, s1)
v2 = sent_vec(nlp, s2)
v3 = sent_vec(nlp, s3)

print("similar (s1, s2):", round(cosine(v1, v2), 4))
print("less similar (s1, s3):", round(cosine(v1, v3), 4))
print("More similar (s2, s3):", round(cosine(v2, v3), 4))

similar (s1, s2): 0.7712
less similar (s1, s3): 0.7148
More similar (s2, s3): 0.8452


## 6) Cosine similarity with sklearn

Now we do the same similarity check using `sklearn.metrics.pairwise.cosine_similarity`.

In [70]:
from sklearn.metrics.pairwise import cosine_similarity

# Reuse the same three example sentences
s1 = "I love this movie."
s2 = "This film is fantastic."
s3 = "The weather is very hot today."

In [71]:
v1 = sent_vec(nlp, s1)
v2 = sent_vec(nlp, s2)
v3 = sent_vec(nlp, s3)

In [72]:
# sklearn expects 2D arrays: shape (n_samples, n_features)
sim_12 = cosine_similarity(v1.reshape(1, -1), v2.reshape(1, -1))[0, 0]
sim_13 = cosine_similarity(v1.reshape(1, -1), v3.reshape(1, -1))[0, 0]
sim_23 = cosine_similarity(v2.reshape(1, -1), v3.reshape(1, -1))[0, 0]

print("sklearn cosine (s1, s2):", round(float(sim_12), 4))
print("sklearn cosine (s1, s3):", round(float(sim_13), 4))
print("sklearn cosine (s2, s3):", round(float(sim_23), 4))

sklearn cosine (s1, s2): 0.7712
sklearn cosine (s1, s3): 0.7148
sklearn cosine (s2, s3): 0.8452


## Quick recap

- Transformer output is in doc._.trf_data
- In new version, use last_hidden_layer_state.data in older versions use doc.tensor
- Shape (N, 768) means N wordpieces, each with 768 features
- You can average vectors for sentence meaning

---
---
---

# Step-by-step: Text format classification using spaCy transformer embeddings

This section mirrors the same constraints:
- input columns: `full_item`, `stage1_code`, `stage2_code`
- preprocessed column: `combine_text`
- combined target: `combined_target`
- privacy-safe labels: `category_stage1`, `category_stage2`, `category_combined`

Implementations included:
1. Two-stage classifier (stage-1 and stage-2 separately)
2. Single-stage classifier (combined target)
3. Confidence-threshold hybrid (sparse base model + embedding refiner)

## Step 1: Imports

In [73]:
import re
import time
import numpy as np
import pandas as pd
import spacy

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score

## Step 2: Load data and preprocess text

In [74]:
df_raw = pd.read_csv("../data/text_format_raw_training_data.csv")

required_cols = ["full_item", "stage1_code", "stage2_code"]
missing = [c for c in required_cols if c not in df_raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df_raw[required_cols].dropna().drop_duplicates().reset_index(drop=True)

# light NLP preprocessor for lemmatization + stopword removal
nlp_pre = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def normalize_2digit(value: str, pick: str = "first") -> str:
    parts = re.findall(r"\d{2}", str(value))
    if not parts:
        digits = re.sub(r"\D", "", str(value))
        if len(digits) >= 2:
            return digits[:2]
        return digits.zfill(2)
    if pick == "second":
        return parts[1] if len(parts) > 1 else parts[0]
    return parts[0]

def preprocess_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    doc = nlp_pre(text)
    tokens = [
        tok.lemma_.strip()
        for tok in doc
        if tok.is_alpha and not tok.is_stop and tok.lemma_.strip()
    ]
    return " ".join(tokens)

df["stage1_code"] = df["stage1_code"].astype(str).apply(lambda x: normalize_2digit(x, pick="first"))
df["stage2_code"] = df["stage2_code"].astype(str).apply(lambda x: normalize_2digit(x, pick="second"))
df["combined_target"] = df["stage1_code"] + " " + df["stage2_code"]

df["combine_text"] = df["full_item"].astype(str).apply(preprocess_text)
df = df[df["combine_text"].str.len() > 0].reset_index(drop=True)

# privacy-safe category labels
stage1_unique = sorted(df["stage1_code"].unique())
stage2_unique = sorted(df["stage2_code"].unique())
stage1_map = {code: f"category_stage1_{i:02d}" for i, code in enumerate(stage1_unique, start=1)}
stage2_map = {code: f"category_stage2_{i:03d}" for i, code in enumerate(stage2_unique, start=1)}

df["category_stage1"] = df["stage1_code"].map(stage1_map)
df["category_stage2"] = df["stage2_code"].map(stage2_map)
df["category_combined"] = df["category_stage1"] + "__" + df["category_stage2"]

category_stage1_to_code = {v: k for k, v in stage1_map.items()}
category_stage2_to_code = {v: k for k, v in stage2_map.items()}

output_path = "../data/text_format_training_data_preprocessed_spacy_trf.csv"
df.to_csv(output_path, index=False)

print("Rows after cleanup:", len(df))
print("Saved transformed data:", output_path)
df[["full_item", "combine_text", "stage1_code", "stage2_code", "combined_target"]].head(3)

Rows after cleanup: 2389
Saved transformed data: ../data/text_format_training_data_preprocessed_spacy_trf.csv


,full_item,combine_text,stage1_code,stage2_code,combined_target
0,Subcontractor bid for condensing high-efficien...,subcontractor bid condense high efficiency boi...,23,70,23 70
1,SCOPE OF WORK DESCRIPTION: Supply and install ...,scope work description supply install evaporat...,23,60,23 60
2,VENDOR SUBMITTAL: Material submittal for veget...,vendor submittal material submittal vegetate g...,07,50,07 50


## Step 3: Train/test split

In [75]:
X = df["combine_text"].tolist()
y_stage1 = df["category_stage1"].tolist()
y_stage2 = df["category_stage2"].tolist()
y_combined = df["category_combined"].tolist()

(
    X_train,
    X_test,
    y_stage1_train,
    y_stage1_test,
    y_stage2_train,
    y_stage2_test,
    y_combined_train,
    y_combined_test,
) = train_test_split(
    X,
    y_stage1,
    y_stage2,
    y_combined,
    test_size=0.2,
    random_state=42,
    stratify=y_combined,
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 1911
Test size: 478


## Step 4: Generate spaCy transformer embeddings from `combine_text`

Uses `en_core_web_trf` and mean-pools the transformer wordpiece vectors.

In [76]:
nlp_embed = spacy.load(
    "en_core_web_trf",
    exclude=["tagger", "parser", "attribute_ruler", "lemmatizer", "ner"],
)

def spacy_trf_embed_texts(texts, batch_size=32):
    vectors = []
    for d in nlp_embed.pipe(texts, batch_size=batch_size):
        ragged = d._.trf_data.last_hidden_layer_state
        vec = ragged.data.mean(axis=0)
        vectors.append(vec)
    return np.vstack(vectors)

start = time.perf_counter()
X_train_emb = spacy_trf_embed_texts(X_train, batch_size=32)
X_test_emb = spacy_trf_embed_texts(X_test, batch_size=32)
embed_time = time.perf_counter() - start

print("Train embedding shape:", X_train_emb.shape)
print("Test embedding shape:", X_test_emb.shape)
print("Embedding generation time (sec):", round(embed_time, 2))

Train embedding shape: (1911, 768)
Test embedding shape: (478, 768)
Embedding generation time (sec): 102.02


## Implementation 1: Two-stage model (stage-1 and stage-2 trained separately)

In [77]:
model_stage1 = LogisticRegression(max_iter=3000)
model_stage2 = LogisticRegression(max_iter=3000)

model_stage1.fit(X_train_emb, y_stage1_train)
model_stage2.fit(X_train_emb, y_stage2_train)

pred_stage1 = model_stage1.predict(X_test_emb)
pred_stage2 = model_stage2.predict(X_test_emb)

print("Two-stage stage-1 accuracy:", round(accuracy_score(y_stage1_test, pred_stage1), 4))
print("Two-stage stage-2 accuracy:", round(accuracy_score(y_stage2_test, pred_stage2), 4))

print("\nStage-1 report")
print(classification_report(y_stage1_test, pred_stage1, zero_division=0))

print("\nStage-2 report")
print(classification_report(y_stage2_test, pred_stage2, zero_division=0))

Two-stage stage-1 accuracy: 0.9665
Two-stage stage-2 accuracy: 0.9247

Stage-1 report
                    precision    recall  f1-score   support

category_stage1_01       0.98      0.97      0.97        60
category_stage1_02       0.95      0.98      0.97        60
category_stage1_03       1.00      0.92      0.96        60
category_stage1_04       0.95      0.97      0.96        59
category_stage1_05       0.98      0.97      0.97        60
category_stage1_06       0.93      0.95      0.94        60
category_stage1_07       0.97      0.98      0.98        60
category_stage1_08       0.97      1.00      0.98        59

          accuracy                           0.97       478
         macro avg       0.97      0.97      0.97       478
      weighted avg       0.97      0.97      0.97       478


Stage-2 report
                     precision    recall  f1-score   support

category_stage2_001       1.00      0.85      0.92        20
category_stage2_002       0.88      0.96      0.92  

## Implementation 2: Single-stage model (combined target)

In [78]:
model_single = LogisticRegression(max_iter=3000)
model_single.fit(X_train_emb, y_combined_train)

pred_combined = model_single.predict(X_test_emb)
print("Single-stage combined accuracy:", round(accuracy_score(y_combined_test, pred_combined), 4))
print(classification_report(y_combined_test, pred_combined, zero_division=0))

pred_single_stage1 = [x.split("__")[0] for x in pred_combined]
pred_single_stage2 = [x.split("__")[1] for x in pred_combined]

print("\nSingle-stage decoded stage-1 accuracy:", round(accuracy_score(y_stage1_test, pred_single_stage1), 4))
print("Single-stage decoded stage-2 accuracy:", round(accuracy_score(y_stage2_test, pred_single_stage2), 4))

Single-stage combined accuracy: 0.954
                                         precision    recall  f1-score   support

category_stage1_01__category_stage2_002       0.89      0.85      0.87        20
category_stage1_01__category_stage2_003       0.86      0.90      0.88        20
category_stage1_01__category_stage2_004       0.95      0.90      0.92        20
category_stage1_02__category_stage2_002       0.91      1.00      0.95        20
category_stage1_02__category_stage2_004       1.00      0.95      0.97        20
category_stage1_02__category_stage2_006       0.95      0.90      0.92        20
category_stage1_03__category_stage2_002       0.91      1.00      0.95        20
category_stage1_03__category_stage2_003       1.00      0.95      0.97        20
category_stage1_03__category_stage2_006       1.00      0.95      0.97        20
category_stage1_04__category_stage2_002       0.95      1.00      0.97        19
category_stage1_04__category_stage2_006       1.00      0.90      0.95

## Confidence-threshold hybrid refinement

Base model uses sparse text features (`HashingVectorizer + LinearSVC + CalibratedClassifierCV`).
For low-confidence samples, fallback to embedding-based refiner.

In [79]:
base_pipe = Pipeline(
    steps=[
        ("vect", HashingVectorizer(n_features=2**18, alternate_sign=False, norm="l2")),
        (
            "clf",
            CalibratedClassifierCV(
                estimator=LinearSVC(),
                method="sigmoid",
                cv=3,
            ),
        ),
    ]
)

base_pipe.fit(X_train, y_combined_train)
base_proba = base_pipe.predict_proba(X_test)
base_pred = base_pipe.classes_[np.argmax(base_proba, axis=1)]
base_conf = np.max(base_proba, axis=1)

emb_refiner = LogisticRegression(max_iter=3000)
emb_refiner.fit(X_train_emb, y_combined_train)
emb_pred = emb_refiner.predict(X_test_emb)

CONF_THRESHOLD = 0.80
final_pred = np.where(base_conf >= CONF_THRESHOLD, base_pred, emb_pred)

print("Base-only combined accuracy:", round(accuracy_score(y_combined_test, base_pred), 4))
print("Hybrid combined accuracy:", round(accuracy_score(y_combined_test, final_pred), 4))
print("Routed to embedding (%):", round(float((base_conf < CONF_THRESHOLD).mean() * 100), 2))

def predict_hybrid_spacy_trf(text: str, threshold: float = 0.80):
    t = preprocess_text(text)
    base_p = base_pipe.predict_proba([t])[0]
    idx = int(np.argmax(base_p))
    base_label = base_pipe.classes_[idx]
    conf = float(base_p[idx])

    if conf >= threshold:
        final_label = base_label
        source = "base"
    else:
        emb = spacy_trf_embed_texts([t])
        final_label = emb_refiner.predict(emb)[0]
        source = "embedding_refiner"

    stage1_cat, stage2_cat = final_label.split("__")

    stage1_code = category_stage1_to_code.get(stage1_cat)
    if stage1_code is None:
        m = df.loc[df["category_stage1"] == stage1_cat, "stage1_code"]
        stage1_code = str(m.iloc[0]) if len(m) else "00"

    stage2_code = category_stage2_to_code.get(stage2_cat)
    if stage2_code is None:
        m = df.loc[df["category_stage2"] == stage2_cat, "stage2_code"]
        stage2_code = str(m.iloc[0]) if len(m) else "00"

    stage1_code = str(stage1_code).zfill(2)
    stage2_code = str(stage2_code).zfill(2)
    combined_target = f"{stage1_code} {stage2_code}"

    return {
        "final_label": final_label,
        "pred_stage1": stage1_code,
        "pred_stage2": stage2_code,
        "pred_combined_target": combined_target,
        "base_confidence": round(conf, 4),
        "prediction_source": source,
    }

example = "provide and install ventilation ductwork and balancing"
print(predict_hybrid_spacy_trf(example, threshold=CONF_THRESHOLD))

Base-only combined accuracy: 0.9979
Hybrid combined accuracy: 0.9916
Routed to embedding (%): 1.46
{'final_label': np.str_('category_stage1_07__category_stage2_004'), 'pred_stage1': '23', 'pred_stage2': '30', 'pred_combined_target': '23 30', 'base_confidence': 0.8711, 'prediction_source': 'base'}


## Benchmark: LogisticRegression vs LinearSVC+Calibrated (spaCy transformer embeddings)

In [80]:
results = []

models = {
    "logistic_regression": LogisticRegression(max_iter=3000),
    "linear_svc_calibrated": CalibratedClassifierCV(
        estimator=LinearSVC(),
        method="sigmoid",
        cv=3,
    ),
}

for name, clf in models.items():
    t0 = time.perf_counter()
    clf.fit(X_train_emb, y_combined_train)
    train_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    pred = clf.predict(X_test_emb)
    pred_time = time.perf_counter() - t1

    acc = accuracy_score(y_combined_test, pred)
    f1 = f1_score(y_combined_test, pred, average="macro", zero_division=0)

    results.append(
        {
            "model": name,
            "accuracy": round(float(acc), 4),
            "macro_f1": round(float(f1), 4),
            "train_time_sec": round(float(train_time), 4),
            "predict_time_sec": round(float(pred_time), 4),
        }
    )

benchmark_df = pd.DataFrame(results).sort_values(["macro_f1", "accuracy"], ascending=False)
print(benchmark_df.to_string(index=False))

                model  accuracy  macro_f1  train_time_sec  predict_time_sec
linear_svc_calibrated    0.9895    0.9896          7.2637            0.0093
  logistic_regression    0.9540    0.9540          2.7764            0.0015
